an obj can go out of scope temporarily, eg if i made a call to another func, but my original func is still open, i haven't hit } yet  
or can be destroyed, if its scope is closed with }  
if the obj is 'static', then even if closing with }, it don't get destroyed  

vars in the stack are destroyed when the function closes  
but vars in the heap aren't, those must be destroyed manually  

### What it a scope

**Scope region**

an area of source code where all declared identifiers are considered distinct from names declared in other scopes  
generally, use indentation to represent scope regions  

being in a region is said 'is in scope'  
being outside of it is said 'being outside of scope'  

**Intuitively: regions**

the code is divided in regions   
regions can be nested (eg. a qhile loop inside a func)  
each region can only see the onject that were declared inside itself or a parent region  

**Easy explanation: identifiers and list**

each region keep a list, of all the thing it declares  
the list has all lines: identifier -> memoryAddress   

rem, identifier can be the name of a var, a func, a class...   
an indentifier is just a nickname for a memory address, valid only for that func that defined it (in that scope)  
instead of telling the code where an obj is, i tell it its nickname, and he finds its address from the list  

technically, all code can read and write in all parts of the memory  
but only the regions which have the identifier in their list know in which part of the meomry that object is   
if tell a region an identifier which is not in its list, he doesn't know where to look for it  

**Nomenclature**

lifetime: runtime property  
scope: compile-time property  

**Out of scope vs destroyed**

out of scope: currently you cannot call it, but later it could come back in scope  
eg. the func where it's in scope made a function call, so it's temporarily out of scope  

destroyed: it won't be in scope ever again  
eg. i returned the func or exited the loop that created it  
eg. i destroyed an instance of a class  

### Working with scope

**How to split in scope regions**

each brackets {} delimitate a region (alco called block, or compound statement, or block statement)  
if the statement i'm executing is inside the {}, i'm in that scope region   
(imagine to follow along the program, seeing at which statement i am, like when using GDB)  

1- funcs:  
each func is a DISJOINT region, bc cannot have nested funcs  
example: main() calls funcA()  
i exit the brackets of main(), and enter the brackets of funcA() (bc cannot be nested)  
now funcA is in scope, main is out of scope  
when funcA returns, i exit the brackets of funcA, enter the brackets of main  
so now main is in scope, funcA out of scope    

2- loops (for, while):  
each loop is a region, bc uses brackets   
but are not disjoint, bc it's written inside the brackets of the function i'm in  
so i'm at the same time in the scope of the loop and of the function  
when i exit the loop, the loop is out of scope, the func is still in scope  
example: main() starts a if() {} clause    
i'm inside the 'if' brackets, but also inside the 'main' brackets, so both are in scope  
when i exit 'if', now 'if' is out of scope, main is still in scope  

**Defininf vars with repeating names**

1- in two disjoint scopes: can use the same identifier  
they're both a line identifier -> address, but each belong to the list of a different region  
so the same identifier mean different things depending on who i tell it to  

2- in two nested scopes: can use the same identifier (but better not to) -> it's called VARIABLE SHADOWING, or NAME HIDING   
while i'm in the inner scope, technically both are in scope, but the identifier defined in the most inner scope takes precedence   
changing the var in this inner scope don't change the var in the outside scope, bc the address in the of that region is differnt  
when i get out of the inner scope, and my local var gets detroyed, i get back my original var in the outer scope  
to read the list: can read the lists of all the parent scopes  
to write in the list: new vars only get written in the list of the innermost scope  

3- in the same scope: cannot use the same identifier twice  
except for overloading a function   

**Example**

1- line A:  
in scope of funcA  
outside of scope of main  
which list can read: funcA, global     
which list can wrte: funcA  

2- line B  
in scope of main() and if()  
which list can read: if(), main(), global   
which list can wite: if()  
read from all parent scopes, write only in innermost  

3- line C  
in scope of main()  
out of scope of if()  
which list can read: main(), global    
which list can wite: main()  

4- line D  
in global scope, so in scope of everything (that comes after line D)  
which list can read: the global one  
which list can write: the global one 

which list can read: which identifier can use in the code  
which list to write: where is saved an identifier declared here  

**Example**

so if declare a var in line B, it gets only written in list of if()    
then, at line C, cannot read list of if()  
so cannot use that var  

**Explanation !!!**

1- what are lists  
identifiers (name of var, name of func) are nicknames for a memory cell  
so each scope has a list:  
identifier -> memoryAddress, varType  
can use an identifier only if, in one of the lists i can currently read, appears such identifier  
so, in different lists, a same identifier can be mean different memory addresses  

2- how to write in list   
when i declare something, it gets written in the list of ONLY the innermost scope region  

3- how do i read from list  
when looking for an identifier, can look at the lists of ALL THE PARENTS' scope regions   
if the identifier appears in more lists, read the one from the innermost region      

**Easy explanation identifier**

i declare an idenifier (var, func, class...)  
instead of telling me 'hey, i put it in the memory cell of address 0x100'  
it tells me 'hey, i put it in the memory cell of address 0x100, but you don't hav to remember that. we can decide to nichname that cell IDENTIFIER, so each time you want this object, ask me for the stuff stored in IDENTIFIER instead. loo, il writte "IDENTIFIER -> 0x100, int" in my personal list  

but this list is stored by the current scope  
the other scopes cannot read it  
so if i go out of scope and ask for IDENTIFIER, the program looks at its curnt lists and cannot find it, and tells me: 'we don't have any memory cells nicknamed like that'  
obv, cell with address 0x100 is still there, and its contents are still the same, but i can't find it by giving the nickname, i need to provide the "official" memory address instead    
and must also remember what type of value i stored in it, since that too was written in the list (which is why pointers have type, to know how to read the value they're pointing to) 

if i do provide the address (pass by pointer), then i'm can modify the value in that cell  
so, when the nickname A returns in scope, and i check the momory cell, i find out that the value has been changed  

***Recap**

an object is in scope:  
from: its definition (NOT declaration)
until: the scope region (block) it was defined into ends  

### Passing stuff between scopes

**Passing vars between funcs**

1- passing it whe i'm calling a func: put it in the args  

2- passing it whe i'm closing a func: put it in the return value  
make sure the return type is correct  

**Classes and funcs**

if a class is defined inside a func, it can only be used inside that func (it's in the scope of that func)
so generally, define all classes outside of main  

objects too, one created in a func is only in scope inside that func  

so it's important to pass stuff between funcs  

### Good practices

vars should be declared in the most inner block possible  
so last only the time strictly necessary  
so, i limit The number of vars in scope at any moment  

since vars have scope inside {}, i could create a block just to limit the scope of some var  
 
but it's bad  
do not create new blocks whose only purpose is to limit the scope of variables  
if want to do that, put that code in a function instead  

exception: can do it if need an index for a loop  

{ // the block here ensures block scope for i
    int i{ 1 };   
    while (i <= 10) 
    {
        std::cout << i << ' ';   
        ++i;
    }
}

### Storage classes (types of scope)  

**Types of scope**

1- local / block scope  
visible from point of declaration until next closing brace }  

2- file / global scope  
visible from point of declaration until end of file  

3- function scope  
visible in all function, even before the point of declaration  
eg. used in goto jumps, so can jumo even to lines i haven't reached yet  

4- loop scope  
for (initialization; test condition; update)
the var defined in 'initialization' have 'loop scope'  
so, scope from the point of definition until the end of the loop  
so, don't get destroyed until i have exited the loop (not after the first repetition, after exiting it completely )

**Assigning scope to a var**

1- default vars  
have block scope, in the block they're defined in  

2- global  
have global scope  

3- keyword: static   
keeps its value between different function calls, the initialization line is only run the first time    
explanation: when func ends, it goes out of scope, but is not deleted (unlike a default var)

4- keyword: extern  
declare a var defined in another source file (external linkage)

4- keyword: mutable  
only can use in members of a class  
allows to modify its value, even if the obj is const  

### Lifetime

**How to destroy an object**

1- obj in stack (local var)  
destroyed is automatically called when obj goes out of scope (when meet })  

2- heap (pointer with new)  
desroyed must be manually called with the command delete  
if i don't, get memory leak  

### Scope vs lifetime

what | lifetime | scope
---|---|---
meaning | when the var 'exist' in memory (hasn't been destroyed yet) | when the var is 'seen' (can use its identifier)  

**Lifetime**

1- global var: static duration  
created at start of program  
destroyed at end of main (end of program)  

2- local var: automatic duration  
created at {  
destroyed at }  


**Scope**

1- local scope  
vars declared inside a func  

2- global scope  
vars declared outside all funcs  